# Fine-Tuning a Qwen 1.5 Model and Logging to Artifactory

This notebook fine-tunes a small Qwen model (`Qwen/Qwen1.5-0.5B-Chat`) on a DevOps instruction dataset using LoRA, then logs the model to JFrog Artifactory.

**Workflow:**
1. **Setup** – Imports and configuration (HF_ENDPOINT, HF_TOKEN from env)
2. **Configuration** – Model, dataset, LoRA, and training parameters
3. **Data Preparation** – Load `Szaid3680/Devops`, format for instruction tuning
4. **Model Loading and Fine-Tuning** – Load base model, apply LoRA, train with SFTTrainer
5. **Evaluation** – Compare fine-tuned vs base model on a sample prompt
6. **Model Logging** – Upload to Artifactory via `frogml.huggingface.log_model` with timestamped version

## 1. Imports

In [14]:
import os
from pathlib import Path

# Load .env FIRST – HF_ENDPOINT must be set before huggingface_hub is imported
from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env", override=True)  # override=True picks up .env changes on re-run

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from trl import SFTTrainer
import frogml

## 2. Configuration

Set HF_ENDPOINT and HF_TOKEN (or HUGGINGFACE_HUB_TOKEN) in `notebooks/.env` for Artifactory HuggingFace proxy. See [Connect Hugging Face to Artifactory](https://jfrog.com/help/r/jfrog-artifactory-documentation/connect-hugging-face-to-artifactory).

In [15]:
import os

# HuggingFace via Artifactory proxy – set HF_ENDPOINT and HF_TOKEN in notebooks/.env
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "86400")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "86400")
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if hf_token:
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

# JFrog repository for uploading the fine-tuned model (override via JF_MODEL_REPO)
JF_MODEL_REPO = os.environ.get("JF_MODEL_REPO", "devops-helper-llms")

In [3]:
# Model, Tokenizer and Dataset configuration
HUGGINGFACE_MODEL = "Qwen/Qwen1.5-0.5B-Chat"
DATASET_NAME = "Szaid3680/Devops"
new_model_adapter = "qwen-0.5b-devops-adapter"

# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./qwen-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=1,
    eval_strategy="steps",
    eval_steps=1,
    fp16=False,  # False for CPU/MPS
)

## 3. Data Preparation

Load `Szaid3680/Devops`, split train/eval, and define the instruction format.

In [4]:
dataset = load_dataset(DATASET_NAME, split="train", token=hf_token)
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# Small subset for demo
train_dataset = train_dataset.select(range(2))
eval_dataset = eval_dataset.select(range(2))

def format_instruction(example):
    instruction = example.get("Instruction", "")
    inp = example.get("Prompt", "")
    response = example.get("Response", "")
    return f"<s>[INST] {instruction}\n{inp} [/INST] {response} </s>"

print("Sample:", train_dataset[0])

INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/datasets/Szaid3680/Devops/resolve/main/README.md "HTTP/1.1 404 "
INFO:httpx:HTTP Request: GET https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/api/datasets/Szaid3680/Devops "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/datasets/Szaid3680/Devops/resolve/9995882cf6dc47a0d41787bb4bdafc38937dbe5c/Devops.py "HTTP/1.1 404 "
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Szaid3680/Devops/Szaid3680/Devops.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/datasets/Szaid3680/Devops/resolve/9995882cf6dc47a0d41787bb4bdafc38937dbe5c/README.md "HTTP/1.1 404 "
INFO:httpx:HTTP Request: GET https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/api/datasets/Szaid3680/Devops/revision/9995882cf6d

Sample: {'Response': "\nUnless you forced a pull you should have been asked by git to commit or stash any changes, so you should be able to revert back to that commit, or unstash what you stashed.\nIf you forced a pull or rebase with uncommited and changes not push to remote branch you are not using git correctly and I suggest reading through some git documentation and doing some tutorials to understand generally accepted practices so you don't encounter data loss in the future.\n", 'Instruction': '\n\n\n\n\n\n\n\nClosed. This question needs details or clarity. It is not currently accepting answers.\n                                \n                            \n\n\n\n\n\n\n\n\n\n\n\nWant to improve this question? Add details and clarify the problem by editing this post.\n\n\nClosed last year.\n\n\n\n\n\n\n\n                        Improve this question\n                    \n\n\n\nI had been having issues pushing my files to a repository and it told me to compare using pull. So when 

## 4. Model Loading and Fine-Tuning

In [5]:
tokenizer = AutoTokenizer.from_pretrained(HUGGINGFACE_MODEL, token=hf_token or True)
tokenizer.pad_token = tokenizer.eos_token

INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/tokenizer_config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: GET https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/api/models/Qwen/Qwen1.5-0.5B-Chat/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 "
INFO:httpx:HTTP Request: GET https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/api/models/Qwen/Qwen1.5-0.5B-Chat/tree/main?recursive=true&expand=false "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/added_tokens.json "HTTP/1.1 404 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    HUGGINGFACE_MODEL,
    device_map="cpu",
    token=hf_token or True,
)
model = get_peft_model(model, lora_config)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    formatting_func=format_instruction,
    args=training_args,
)

print("--- Starting Fine-Tuning ---")
train_output = trainer.train()
print("--- Fine-Tuning Complete ---")

INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/adapter_config.json "HTTP/1.1 404 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/adapter_config.json "HTTP/1.1 404 "


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/generation_config.json "HTTP/1.1 200 "
/opt/anaconda3/envs/devops-helper/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/processor_config.json "HTTP/1.1 404 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/preprocessor_config.json "HTTP/1.1 404 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/preprocessor_config.json "HTTP/1.1 404 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/tokenizer_config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/con

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


--- Starting Fine-Tuning ---


Step,Training Loss,Validation Loss
1,No log,3.032140


INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "


--- Fine-Tuning Complete ---


## 5. Evaluation

In [7]:
# Use metrics from training (avoids NotebookProgressCallback error when calling evaluate() separately)
metrics = train_output.metrics
print("--- Evaluation Metrics ---")
print(metrics)

--- Evaluation Metrics ---
{'train_runtime': 1167.83, 'train_samples_per_second': 0.007, 'train_steps_per_second': 0.001, 'total_flos': 1247246284800.0, 'train_loss': 4.772970676422119}


In [8]:
trainer.model.save_pretrained(new_model_adapter)

INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "


In [9]:
base_model = AutoModelForCausalLM.from_pretrained(HUGGINGFACE_MODEL, device_map="cpu", token=hf_token or True)
finetuned_model = PeftModel.from_pretrained(base_model, new_model_adapter)
finetuned_model = finetuned_model.merge_and_unload()

prompt = "How do I expose a deployment in Kubernetes using a service?"
messages = [
    {"role": "system", "content": "You are a helpful DevOps assistant."},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to("cpu")
input_ids_len = model_inputs["input_ids"].shape[1]

print("--- FINE-TUNED MODEL ---")
generated_ids = finetuned_model.generate(model_inputs.input_ids, max_new_tokens=256)
response_finetuned = tokenizer.decode(generated_ids[:, input_ids_len:][0], skip_special_tokens=True)
print(response_finetuned)

print("\n--- BASE MODEL ---")
original_model = AutoModelForCausalLM.from_pretrained(HUGGINGFACE_MODEL, device_map="cpu", token=hf_token or True)
generated_ids_base = original_model.generate(model_inputs.input_ids, max_new_tokens=256)
response_base = tokenizer.decode(generated_ids_base[:, input_ids_len:][0], skip_special_tokens=True)
print(response_base)

INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/adapter_config.json "HTTP/1.1 404 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/adapter_config.json "HTTP/1.1 404 "


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/generation_config.json "HTTP/1.1 200 "
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- FINE-TUNED MODEL ---
To expose a deployment in Kubernetes using a service, you can use the `kubectl` command-line tool to create an API resource that defines the resources and behavior of the deployment.
Here is an example of how you might create a Kubernetes deployment that exposes a RESTful API:
```bash
$ kubectl apply -f api.yaml

apiVersion: v1
kind: Deployment
metadata:
  name: my-deployment
spec:
  replicas: 3
  selector:
    matchLabels:
      app: my-app
  template:
    metadata:
      labels:
        app: my-app
    spec:
      containers:
      - name: my-container
        image: my-image
        ports:
        - containerPort: 80
```

In this example, we created a new Kubernetes deployment with three replicas (using the `replicas` key), a label that specifies that our deployment should only be accessible by running the `my-app` app, and a template that describes the API resource being exposed.
When we run the `kubectl apply` command, it will create a new API resource cal

INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/adapter_config.json "HTTP/1.1 404 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/config.json "HTTP/1.1 200 "
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/adapter_config.json "HTTP/1.1 404 "


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
INFO:httpx:HTTP Request: HEAD https://tomjfrog.jfrog.io/artifactory/api/huggingfaceml/hf-remote/Qwen/Qwen1.5-0.5B-Chat/resolve/main/generation_config.json "HTTP/1.1 200 "


To expose a deployment in Kubernetes using a service, you can follow these steps:

1. Identify the services that you want to expose. You will need to choose which services your application needs to be running on.

2. Create a Kubernetes service for each of the services you identified. You can use a tool like `kubectl create-service` to create a new service and specify its name, labels, and runtime type.

3. Deploy the services as replicas of the original deployment. To do this, you will need to deploy the services with their respective replicas to a Kubernetes cluster.

4. In your production environment, run the services to ensure that they are functioning properly.

5. When deploying the services in a production environment, make sure that the cluster is up and running by running any necessary commands or by checking the status of the service through a monitoring tool.

6. Once you have deployed the services, you can start to monitor them for issues and troubleshoot if needed. This wi

## 6. Model Logging

Log the fine-tuned model to Artifactory with a timestamped version.

In [16]:
from datetime import datetime

version = f"0.0.1-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

try:
    frogml.huggingface.log_model(
        model=finetuned_model,
        tokenizer=tokenizer,
        repository=JF_MODEL_REPO,
        model_name="devops_helper",
        version=version,
        parameters={"finetuning-dataset": DATASET_NAME},
        code_dir="code_dir",
        dependencies=["main/conda.yaml"],
        metrics=metrics,
        predict_file="code_dir/predict.py",
    )
    print(f"--- Model Logged Successfully (version {version}) ---")
except Exception as e:
    print(f"Model logging failed: {e}")

INFO:HuggingfaceModelVersionManager:Logging model devops_helper to devopshelper-ml-local


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:JmlCustomerClient:Customer exists in JML.
INFO:frogml.sdk.model_version.utils.files_tools:Code directory, predict file and dependencies are provided. Setup template files for model_name devops_helper


/private/var/folders/k2/2jpmnnl51yqf5vqb6yz486hc0000gn/T/tmp3gwzmq5m/devops_helper.pretrained_model/tokenizer_…

/private/var/folders/k2/2jpmnnl51yqf5vqb6yz486hc0000gn/T/tmp3gwzmq5m/devops_helper.pretrained_model/config.jso…

...

/private/var/folders/k2/2jpmnnl51yqf5vqb6yz486hc0000gn/T/tmp3gwzmq5m/devops_helper.pretrained_model/chat_templ…

/private/var/folders/k2/2jpmnnl51yqf5vqb6yz486hc0000gn/T/tmp3gwzmq5m/devops_helper.pretrained_model/model.safe…

main/conda.yaml:   0%|          | 0.00/283 [00:00<?, ?B/s]

/var/folders/k2/2jpmnnl51yqf5vqb6yz486hc0000gn/T/tmp3gwzmq5m/code.zip:   0%|          | 0.00/2.71k [00:00<?, ?…

2026-03-13 11:55:34,207 - INFO - frogml.storage.logging._log_config.frog_ml.__upload_model:529 - Model: "devops_helper", version: "0.0.1-20260313-115419" has been uploaded successfully
--- Model Logged Successfully (version 0.0.1-20260313-115419) ---
